In [1]:
%load_ext autoreload
%autoreload 2

In [8]:
from sklearn.datasets import make_blobs
from src.models import KMeansMember
from src.ensemble import AdaptiveClusteringEnsemble
from sklearn.metrics import adjusted_rand_score

In [9]:
# Generate a synthetic dataset (n=15 instances, 4 features)
X_synth, true_labels = make_blobs(n_samples=15, centers=3, n_features=4, random_state=42)

# Initialize the separated architecture
kmeans_base = KMeansMember()
baseline_math = BaselineMetrics()

ace_framework = AdaptiveClusteringEnsemble(
    base_member=kmeans_base,
    metrics=baseline_math,
    m_partitions=4,
    k_clusters=3,
    alpha_1=0.8,
    random_state=123
)

# Run Stage 1
print("Running Stage 1: Matrix Transformation...")
theta_1_initial = ace_framework.stage1_transformation(X_synth)
print(f"Generated {theta_1_initial.shape[1]} initial base clusters.")

# Run Stage 2
print("Running Stage 2: Adaptive Consensus Merging...")
theta_1_final, theta_2_elim = ace_framework.stage2_consensus_merging(theta_1_initial, X_synth.shape[0])
print(f"Active consensus clusters remaining: {theta_1_final.shape[1]}")

# Run Stage 3
print("Running Stage 3: Hard Clustering Enforcement...")
final_labels = ace_framework.stage3_hard_clustering(theta_1_final, theta_2_elim)

# Validation against Ground Truth
ari_score = adjusted_rand_score(true_labels, final_labels)
print("\n======================================")
print(f"Final ACE Clustering Complete!")
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
print("======================================")

Running Stage 1: Matrix Transformation...
Generated 12 initial base clusters.
Running Stage 2: Adaptive Consensus Merging...
Active consensus clusters remaining: 3
Running Stage 3: Hard Clustering Enforcement...

Final ACE Clustering Complete!
Adjusted Rand Index (ARI): 1.0000
